# PyPI Package Data Cleaning

In [1]:
import pandas as pd

## Dataset Loading

In [2]:
data = r"..\data\raw\pypi_packages_raw.csv"

In [3]:
pypi_df=pd.read_csv(data)

## Dataset Structure Check


In [4]:
pypi_df.head()

,package_name,version,summary,license,requires_python,downloads_last_day,downloads_last_week,downloads_last_month
0,numpy,2.5.3,Fundamental package for array computing in Python,NaN,>=3.12,21056506,188423904,766927112
1,pandas,3.0.6,"Powerful data structures for data analysis, ti...",BSD 3-Clause License\n \n Copy...,>=3.11,14269563,129551189,525364580
2,scipy,1.18.1,Fundamental algorithms for scientific computin...,"Copyright (c) 2001-2002 Enthought, Inc. 2003, ...",>=3.12,7515812,68946699,286276241
3,matplotlib,3.11.2,Python plotting package,License agreement for matplotlib versions 1.3....,>=3.11,3617048,36000749,156039268
4,seaborn,0.13.2,Statistical data visualization,NaN,>=3.8,579029,6097739,26297024


In [5]:
pypi_df.shape

(32, 8)

In [6]:
pypi_df.columns

Index(['package_name', 'version', 'summary', 'license', 'requires_python',
       'downloads_last_day', 'downloads_last_week', 'downloads_last_month'],
      dtype='str')

In [7]:
pypi_df.dtypes

package_name              str
version                   str
summary                   str
license                   str
requires_python           str
downloads_last_day      int64
downloads_last_week     int64
downloads_last_month    int64
dtype: object

In [8]:
pypi_df.isnull().sum()

package_name             0
version                  0
summary                  0
license                 17
requires_python          0
downloads_last_day       0
downloads_last_week      0
downloads_last_month     0
dtype: int64

In [9]:
pypi_df.duplicated().sum()

np.int64(0)

In [10]:
pypi_df["package_name"].value_counts()


package_name
numpy             1
pandas            1
scipy             1
matplotlib        1
seaborn           1
scikit-learn      1
tensorflow        1
torch             1
transformers      1
statsmodels       1
keras             1
opencv-python     1
nltk              1
Django            1
Flask             1
fastapi           1
requests          1
beautifulsoup4    1
selenium          1
Scrapy            1
jupyterlab        1
pydantic          1
SQLAlchemy        1
pytest            1
rich              1
typer             1
pillow            1
sympy             1
networkx          1
gradio            1
langchain         1
openai            1
Name: count, dtype: int64

## Data Cleaning

In [11]:
pypi_df=pypi_df.rename(
    columns={
        "requires_python": "required_python_version",
        "downloads_last_day": "downloads_daily",
        "downloads_last_week": "downloads_weekly",
        "downloads_last_month": "downloads_monthly"
    }
)

In [12]:
pypi_df["license"] = pypi_df["license"].fillna("Unknown")

## Numerical Analysis

In [13]:
pypi_df.describe()

,downloads_daily,downloads_weekly,downloads_monthly
count,3.200000e+01,3.200000e+01,3.200000e+01
mean,6.541176e+06,6.059815e+07,2.506981e+08
std,7.517833e+06,6.940973e+07,2.853905e+08
min,9.742900e+04,9.741620e+05,4.372611e+06
25%,7.455180e+05,7.406573e+06,3.122145e+07
50%,4.139770e+06,3.804013e+07,1.627028e+08
75%,8.504951e+06,8.025904e+07,3.281748e+08
max,2.999608e+07,2.815894e+08,1.167999e+09


In [14]:
pypi_df.select_dtypes(int).agg(["mean","median","min","max","std",'var']).rename(index={"mean":"average","min":"minimum","max":"maximum","std":"standard deviation","var":"variance"})

,downloads_daily,downloads_weekly,downloads_monthly
average,6.541176e+06,6.059815e+07,2.506981e+08
median,4.139770e+06,3.804013e+07,1.627028e+08
minimum,9.742900e+04,9.741620e+05,4.372611e+06
maximum,2.999608e+07,2.815894e+08,1.167999e+09
standard deviation,7.517833e+06,6.940973e+07,2.853905e+08
variance,5.651782e+13,4.817711e+15,8.144772e+16


### Outlier Detection


In [15]:
Q1=pypi_df.select_dtypes(int).quantile(0.25)
Q3=pypi_df.select_dtypes(int).quantile(0.75)
print("Q1:\n",Q1)
print("Q3:\n",Q3)


Q1:
 downloads_daily        745518.0
downloads_weekly      7406573.0
downloads_monthly    31221448.5
Name: 0.25, dtype: float64
Q3:
 downloads_daily      8.504951e+06
downloads_weekly     8.025904e+07
downloads_monthly    3.281748e+08
Name: 0.75, dtype: float64


In [16]:
IQR=Q3-Q1
print("IQR:\n",IQR)

IQR:
 downloads_daily      7.759433e+06
downloads_weekly     7.285247e+07
downloads_monthly    2.969533e+08
dtype: float64


In [17]:
lower_limit=Q1-1.5*IQR
upper_limit=Q3+1.5*IQR
print("lower limit:\n",lower_limit)
print("upper limit:\n",upper_limit)

lower limit:
 downloads_daily     -1.089363e+07
downloads_weekly    -1.018721e+08
downloads_monthly   -4.142085e+08
dtype: float64
upper limit:
 downloads_daily      2.014410e+07
downloads_weekly     1.895377e+08
downloads_monthly    7.736048e+08
dtype: float64


In [27]:
for col in pypi_df.select_dtypes(int).columns:
    outliers = pypi_df[
        (pypi_df[col] < lower_limit[col])
        | (pypi_df[col] > upper_limit[col])
    ]
    print("Outliers in",col,":",len(outliers))

Outliers in downloads_daily : 4
Outliers in downloads_weekly : 2
Outliers in downloads_monthly : 2


## Categorical Data Consistency 

In [19]:
pypi_df["package_name"].nunique() == len(pypi_df["package_name"])


True

In [20]:
having_space=pypi_df["package_name"] != pypi_df["package_name"].str.strip()
having_space.sum()

np.int64(0)

In [21]:
pypi_df["version"].value_counts(ascending=False)

version
2.5.3       1
3.0.6       1
1.18.1      1
3.11.2      1
0.13.2      1
1.9.1       1
2.21.0      1
2.14.0      1
5.17.0      1
0.15.0      1
3.15.1      1
5.0.0.93    1
3.10.3      1
6.1.1       1
3.1.3       1
0.141.1     1
2.34.2      1
4.15.0      1
4.49.0      1
2.19.0      1
4.6.4       1
2.13.5      1
2.1.1       1
9.1.1       1
15.0.0      1
0.27.2      1
12.3.0      1
1.14.0      1
3.7         1
6.28.0      1
1.4.2       1
3.19.2      1
Name: count, dtype: int64

## Python Version Requirement Consistency

In [22]:
pypi_df["required_python_version"].nunique()

11

In [23]:
pypi_df["required_python_version"].unique()


<StringArray>
[         '>=3.12',          '>=3.11',           '>=3.8',          '>=3.10',
        '>=3.10.0',           '>=3.6',           '>=3.9',         '>=3.7.0',
         '>=3.9.0', '!=3.14.1,>=3.12', '<4.0.0,>=3.10.0']
Length: 11, dtype: str

## Download Data Consistency

In [24]:
compare_1=pypi_df["downloads_daily"]>pypi_df["downloads_weekly"]
compare_1.sum()

np.int64(0)

In [25]:
compare_2 = pypi_df["downloads_weekly"] > pypi_df["downloads_monthly"]
compare_2.sum()


np.int64(0)

## Cleaned Dataset Export

In [29]:
pypi_df.to_csv(r"..\data\processed\pypi_packages_cleaned.csv")